# EventPulse — College Event Feedback Analyzer

**Agent Experience Hackathon: Strands Decider 2B**

EventPulse collects student feedback on college events, classifies each response using **Strands Decider 2B** (positive / negative / neutral, and whether the organizer needs to act), and lets organizers query sentiment summaries and top complaints.

The generative conversation runs on **Amazon Bedrock** (Amazon Nova Pro). The classification decision runs in a single forward pass through Strands Decider 2B — no LLM call, calibrated probabilities.

---

## What this notebook shows

1. How the `FeedbackPolicyCheck` decision works — the typed questions, option descriptions, and verdict branching
2. Classifying real feedback samples with Strands Decider 2B directly
3. Running the full `HackathonAgent` end-to-end: submit feedback, get a summary, get top complaints
4. Latency comparison: Strands Decider 2B vs. Amazon Bedrock LLM on the same classification task

> **No API keys in this notebook.** Keys are read from `.env` in the project root (written by `./kit setup`). See the README for setup instructions.

## 0. Setup

In [ ]:
# Standard library + project imports — no pip install needed if you ran ./kit setup
import json
import sys
import time
from pathlib import Path

# Make sure we can import decider_kit and agent from the project root
ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load .env (DECIDER_URL, DECIDER_API_KEY, BEDROCK_MODEL_ID, AWS_PROFILE, ...)
from decider_kit.config import load_dotenv, settings
load_dotenv()

s = settings()
print(f"Decider mode : {s.decider_mode}")
print(f"Decider URL  : {s.decider_url}")
print(f"Bedrock model: {s.bedrock_model}")
print(f"AWS region   : {s.aws_region}")

## 1. The decision: typed questions sent to Strands Decider 2B

Before a feedback response is stored, `FeedbackPolicyCheck` sends a **state** (event name + feedback text) and three typed questions to the decider in a single call:

| Question | Type | What it decides |
|---|---|---|
| `sentiment` | `choice` | positive / negative / neutral |
| `needs_attention` | `noul` | does the organizer need to act? |
| `severity` | `score` | none / minor / serious |

All three are answered in **one forward pass** — the state is encoded once and shared.

In [ ]:
from decider_kit import Decider
from agent.decision import QUESTIONS, build_state, PATTERN_TAG

print(f"Pattern tag : {PATTERN_TAG}")
print()
print("Questions sent to Strands Decider 2B:")
for name, q in QUESTIONS.items():
    qtype = q["type"]
    instruction = q["instructions"]
    criteria = q.get("criteria", {})
    print(f"  [{qtype}] {name}: {instruction}")
    if isinstance(criteria, dict):
        for k, v in criteria.items():
            print(f"             {k!r}: {v}")
    else:
        print(f"             levels: {criteria}")

## 2. Classify individual feedback samples

Let's call Strands Decider 2B directly and see the raw probabilities it returns.

In [ ]:
d = Decider()

samples = [
    ("TechFest 2026",   "The workshops were incredibly well organized. Learned a lot about machine learning!"),
    ("TechFest 2026",   "The microphone kept cutting out during the keynote. Half the audience couldn't hear anything."),
    ("HackMadurai 2026","The internet connection was terrible. We lost almost an hour of hacking time."),
    ("Cultural Night 2026", "The dance performances were breathtaking. Incredibly talented students!"),
    ("Cultural Night 2026", "The stage lighting failed during the classical dance segment. Completely ruined the mood."),
    ("TechFest 2026",   "The event was okay. Nothing particularly stood out to me."),
]

print(f"{'Event':<26} {'Sentiment':<10} {'Conf':>5}  {'Needs Attn':>10}  {'Severity':<8}  {'Latency':>8}  Text")
print("-" * 120)

for event, text in samples:
    state = build_state(event, text)
    t0 = time.perf_counter()
    answers = d.ask(state, QUESTIONS)
    ms = (time.perf_counter() - t0) * 1000

    sentiment     = answers["sentiment"].label
    sent_conf     = answers["sentiment"].confidence
    needs_attn    = answers["needs_attention"].yes
    severity      = answers["severity"].label

    snippet = text[:60] + "..." if len(text) > 60 else text
    print(f"{event:<26} {sentiment:<10} {sent_conf:>5.2f}  {'yes' if needs_attn else 'no':>10}  {severity:<8}  {ms:>7.0f}ms  {snippet}")

## 3. Full verdict logic: what the agent does with each answer

`FeedbackPolicyCheck.classify()` applies thresholds and produces a **verdict** the agent branches on:

| Verdict | Meaning |
|---|---|
| `store` | no action needed; store normally |
| `review` | needs attention but not critical; show to organizer |
| `flag` | serious or negative + needs attention; highlight prominently |
| `unclear` | low confidence (< 0.55); surface for manual review |


In [ ]:
from agent.decision import FeedbackPolicyCheck

checker = FeedbackPolicyCheck(d)

print(f"{'Event':<26} {'Verdict':<10} {'Sentiment':<10} {'Attn':>5}  {'Severity':<8}  Text")
print("-" * 110)

for event, text in samples:
    result = checker.classify(event, text)
    verdict   = result["verdict"]
    sentiment = result["sentiment"]
    needs_attn = result["needs_attention"]
    severity  = result["severity"]
    snippet   = text[:55] + "..." if len(text) > 55 else text

    # Colour-code the verdict for readability
    ICONS = {"store": "green", "review": "yellow", "flag": "red", "unclear": "blue"}
    flag = {"store": "  ", "review": "* ", "flag": "!!", "unclear": "??"}[verdict]
    print(f"{flag} {event:<24} {verdict:<10} {sentiment:<10} {'yes' if needs_attn else 'no':>5}  {severity:<8}  {snippet}")

print()
print("Legend: !! = flag (organizer action needed)  * = review  ?? = unclear (low confidence)")

## 4. Low-confidence handling

When the decider's confidence is below 0.55 the verdict is `unclear`. Those responses are surfaced separately so organizers can review them manually instead of acting on a guess.

In [ ]:
# Ambiguous examples that may land in the unclear bucket
ambiguous = [
    ("TechFest 2026", "The event was okay. Nothing particularly stood out to me."),
    ("HackMadurai 2026", "It was an event. I attended. Some sessions were technical."),
    ("Cultural Night 2026", "Some performances felt too long and repetitive. Could have trimmed a few acts."),
]

for event, text in ambiguous:
    state   = build_state(event, text)
    answers = d.ask(state, QUESTIONS)
    result  = checker.classify(event, text)
    print(f"Text    : {text}")
    print(f"Verdict : {result['verdict']}  (sentiment conf = {result['sentiment_confidence']:.2f}, "
          f"p_attention = {result['p_attention']:.2f})")
    print(f"Low conf: {result['low_confidence']}")
    print()

## 5. Running the full agent

Now we use the `HackathonAgent` — Strands Agents + Amazon Bedrock for the conversation, Strands Decider 2B wired in through tools.

In [ ]:
from agent.agent import HackathonAgent, NAME

print(f"Agent: {NAME}")
agent = HackathonAgent()
print("Agent ready.")

In [ ]:
# Turn 1: list events
out = agent.chat("What events have feedback right now?", session_id="notebook")
print(out["reply"])
print(f"\n[{out['latency_ms']} ms | {len(out['decisions'])} decision(s)]")

In [ ]:
# Turn 2: submit a complaint — this triggers the Strands Decider 2B classification
out = agent.chat(
    "I want to submit feedback for TechFest 2026. "
    "The microphone kept cutting out during the keynote and half the audience couldn't hear a thing.",
    session_id="notebook",
)
print(out["reply"])
print(f"\n[{out['latency_ms']} ms | {len(out['decisions'])} decision(s)]")

if out["decisions"]:
    print("\nDecider output:")
    for dec in out["decisions"]:
        print(f"  pattern   : {dec.get('pattern')}")
        print(f"  sentiment : {dec.get('sentiment')} (conf {dec.get('sentiment_confidence', 0):.2f})")
        print(f"  attention : {dec.get('needs_attention')}  severity: {dec.get('severity')}")
        print(f"  verdict   : {dec.get('verdict')}")
        print(f"  decider   : {dec.get('decider_ms')} ms")

In [ ]:
# Turn 3: submit a positive response
out = agent.chat(
    "Submit feedback for EVT001: Amazing workshops, learned so much about ML!",
    session_id="notebook",
)
print(out["reply"])
print(f"\n[{out['latency_ms']} ms | {len(out['decisions'])} decision(s)]")

if out["decisions"]:
    dec = out["decisions"][0]
    print(f"  sentiment: {dec.get('sentiment')}  verdict: {dec.get('verdict')}  decider: {dec.get('decider_ms')} ms")

In [ ]:
# Turn 4: ask for a full event summary
out = agent.chat(
    "Give me a full summary of the feedback for TechFest 2026.",
    session_id="notebook",
)
print(out["reply"])
print(f"\n[{out['latency_ms']} ms | {len(out['decisions'])} decision(s)]")

In [ ]:
# Turn 5: drill into complaints for a specific event
out = agent.chat(
    "What are the top complaints for Cultural Night 2026?",
    session_id="notebook",
)
print(out["reply"])
print(f"\n[{out['latency_ms']} ms | {len(out['decisions'])} decision(s)]")

## 6. Latency comparison: Strands Decider 2B vs. Bedrock LLM

We ask the same three classification questions using both the decider and the Bedrock LLM, then compare.

In [ ]:
from decider_kit.llm import ask as llm_ask

TEST_CASES = [
    ("TechFest 2026",      "The microphone kept cutting out during the keynote. Half the audience couldn't hear anything."),
    ("HackMadurai 2026",   "Mentors were extremely helpful and available throughout the event."),
    ("Cultural Night 2026","The event ran 2 hours overtime. Parents and students had transport issues."),
    ("TechFest 2026",      "Amazing experience overall! Would definitely attend next year."),
    ("HackMadurai 2026",   "Prizes were not handed out on the day. Still waiting after two weeks."),
]

decider_times, llm_times = [], []

LLM_PROMPT_TEMPLATE = """Classify this college event feedback.
Event: {event}
Feedback: {text}

Answer ONLY with a JSON object with exactly these keys:
  sentiment: one of positive / negative / neutral
  needs_attention: true or false
  severity: one of none / minor / serious"""

print(f"{'#':<3} {'Decider':>8}  {'LLM':>7}  {'Sentiment (D)':>13}  {'Sentiment (L)':>13}  Text")
print("-" * 100)

for i, (event, text) in enumerate(TEST_CASES, 1):
    # --- Strands Decider 2B ---
    state = build_state(event, text)
    t0 = time.perf_counter()
    answers = d.ask(state, QUESTIONS)
    decider_ms = (time.perf_counter() - t0) * 1000
    decider_times.append(decider_ms)

    # --- Bedrock LLM ---
    prompt = LLM_PROMPT_TEMPLATE.format(event=event, text=text)
    t0 = time.perf_counter()
    llm_reply, _ = llm_ask(prompt, max_tokens=80)
    llm_ms = (time.perf_counter() - t0) * 1000
    llm_times.append(llm_ms)

    # Parse LLM JSON (best-effort)
    llm_sentiment = "?"
    try:
        import re
        m = re.search(r'\{.*?\}', llm_reply, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            llm_sentiment = parsed.get("sentiment", "?")
    except Exception:
        pass

    snippet = text[:45] + "..." if len(text) > 45 else text
    print(f"{i:<3} {decider_ms:>7.0f}ms  {llm_ms:>6.0f}ms  {answers['sentiment'].label:>13}  {llm_sentiment:>13}  {snippet}")

In [ ]:
import statistics

d_med = statistics.median(decider_times)
l_med = statistics.median(llm_times)
speedup = l_med / d_med if d_med > 0 else float("inf")

print("Summary")
print("=" * 40)
print(f"Strands Decider 2B  median : {d_med:.0f} ms")
print(f"Bedrock LLM         median : {l_med:.0f} ms")
print(f"Speedup (p50)              : {speedup:.1f}x")
print()
print("Cost per 1,000 decisions (approximate):")
print(f"  Strands Decider 2B (local/hosted): $0.00")
print(f"  Bedrock LLM (Nova Pro est.)       : ~$0.18")

## 7. Classify all pre-loaded feedback in batch

Run the policy classifier over every feedback record loaded from `agent/data/feedback.json` and show the verdict distribution.

In [ ]:
import json
from collections import Counter
from pathlib import Path

data = json.loads((Path("agent/data/feedback.json")).read_text())
events_map = {e["id"]: e["name"] for e in data["events"]}

fresh_checker = FeedbackPolicyCheck(d)
results = []

print(f"Classifying {len(data['feedback'])} feedback records ...\n")

for fb in data["feedback"]:
    event_name = events_map[fb["event_id"]]
    rec = fresh_checker.classify(event_name, fb["text"])
    rec["feedback_id"] = fb["id"]
    rec["event"] = event_name
    rec["text"] = fb["text"]
    results.append(rec)

# Distribution
verdicts   = Counter(r["verdict"] for r in results)
sentiments = Counter(r["sentiment"] for r in results)

print("Verdict distribution:")
for v, n in sorted(verdicts.items()):
    bar = "#" * n
    print(f"  {v:<10} {n:>3}  {bar}")

print()
print("Sentiment distribution:")
for s, n in sorted(sentiments.items()):
    bar = "#" * n
    print(f"  {s:<10} {n:>3}  {bar}")

In [ ]:
# Show all flagged items
flagged = [r for r in results if r["verdict"] == "flag"]
print(f"Flagged responses ({len(flagged)}) — organizer action needed:\n")
for r in flagged:
    print(f"  [{r['event']}] {r['text']}")
    print(f"    -> severity: {r['severity']}  |  decider: {r['decider_ms']} ms")
    print()

In [ ]:
# Show unclear responses
unclear = [r for r in results if r["verdict"] == "unclear"]
if unclear:
    print(f"Unclear responses ({len(unclear)}) — sent for manual review:\n")
    for r in unclear:
        print(f"  [{r['event']}] {r['text']}")
        print(f"    -> sentiment conf: {r['sentiment_confidence']:.2f}  |  p_attention: {r['p_attention']:.2f}")
        print()
else:
    print("No unclear responses in this batch.")

## 8. The reusable decision pattern

The pattern lives in `agent/decision.py`. To reuse it in a different agent:

1. Copy `agent/decision.py`
2. Replace `SENTIMENT_CATEGORIES` and `FEEDBACK_POLICY` with your domain's taxonomy
3. Call `FeedbackPolicyCheck.classify(context, text)` wherever you store or act on user-submitted text
4. Tune `attention_at` and `serious_at` thresholds against your own labelled cases with `./kit bench`

The pattern tag is `#policy`. The three questions (choice + noul + score) together take one forward pass.

In [ ]:
# Print the module docstring — the reusability note shipped with the code
import agent.decision as _dec_mod
print(_dec_mod.__doc__)

---

## Next steps

- `./kit demo` — live chat UI at http://localhost:8501 with the decision panel
- `./kit bench` — measure accuracy and latency of the decider vs. the Bedrock LLM on the labelled eval cases
- `./kit deploy` — deploy to Amazon Bedrock AgentCore Runtime
- `./kit video` — record the demo video for the submission

---

*Built with the [AWS User Group Madurai Strands Decider hackathon kit](https://awsugmdu.in)*